Description:

Analyze the relationship between the number of primary care appointments per head of population in sub-ICBs within SNEE (Suffolk and North East Essex) and staffing levels per head of population. The goal is to use the GP Patient list, the most recent appointments dataset, and the staffing dataset from NHSE (all currently available in the catalog) to investigate how the patient-to-GP ratio correlates with appointments per head specifically in the context of SNEE compared to broader English sub-ICBs.

Exam question/Objective:

Determine if staffing levels per head of population significantly affect the number of primary care appointments per head in sub-ICBs within SNEE. Additionally, assess whether the patient-to-GP ratio in SNEE reflects or deviates from the trends seen in other English sub-ICBs.

Stakeholder Request:

I want a report that details the relationship between the number of primary care appointments per head of population and staffing levels per head of population in sub-ICBs within SNEE. The report should highlight if variations in patient-to-GP ratios contribute to differences in appointments per head and whether these findings are consistent with or diverge from other English sub-ICBs.

Key Data Sources:

GP Patient list from NHSE
Most recent appointments dataset from NHSE
Staffing dataset from NHSE
Methodology/Approach:

Data extraction from NHSE catalog for GP Patient list, appointments data, and staffing data.
Clean and preprocess the datasets to ensure consistency.
Calculate patient-to-GP ratios and staffing levels per head of population.
Conduct initial exploratory data analysis (EDA) to understand distribution and correlations.
Apply statistical analysis (e.g., linear regression or correlation analysis) to assess relationships.
Compare results between SNEE sub-ICBs and other English sub-ICBs.
Visualize findings through charts and graphs.
Summarize insights in a report.
Outputs:

Branch in repo
Notebook
Markdown blog

## Library Imports

In [3]:
import os
from pathlib import Path
if 'notebooks' in str(Path.cwd()):
    os.chdir('..')

# Library imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from matplotlib.ticker import FuncFormatter
import datetime as dt
import pickle
from typing import Dict
from time import sleep

# project imports from src
from src.schemas import DataCatalog
from src.various_methods import PlotCounter, get_workingdays
from src import constants

# Importing SNEE styles
from sneeifstyles import mpl_style
mpl_style()

import warnings
warnings.filterwarnings("ignore")

## Initial Set-up

In [4]:
## Constants
SNEE_SUB_ICB = ['06L','06T','07K']
SNEE_SUB_ICB_NAMES = ['Ipswich & East Suffolk', 'North East Essex', 'West Suffolk']
NOTEBOOK_ALIAS = "Appointments"
GP_PATIENTS_LIST_CATALOG = 'Patients Registered at a GP practice, September 2024'
GP_APPOINTMENTS_CATALOG_NAME:str = 'Appointments in General Practice, August 2024'
GP_WORKFORCE_CATALOG:str = 'General Practice workforce'
GP_LIST_AGE_BANDS = constants.GP_LIST_AGE_BANDS 
GP_LIST_AGE_LABELS = constants.GP_LIST_AGE_LABELS 
SNEE_SUBICB_CODES = list(constants.ONS_CODES.keys())

# Loading the Data Catalog
catalog =  DataCatalog.load_from_yaml("data_catalog.yaml")

# Initializing the plotCounter object
plot_counter = PlotCounter(name=NOTEBOOK_ALIAS)

# set up output directories
for i in ['outputs/assumptions', 'outputs/plots', 'outputs/tables']:
    if not os.path.exists(i):
        os.makedirs(i)

In [ ]:
def process_gp_list(df):
    """
    Filters the given DataFrame by 
    Args:
        df (pandas.DataFrame): GP LIST DataFrame. 
    Returns:
        pandas.DataFrame: 
    """
    df_ = df.copy()
    
    # Keeping only the total population rows
    df_ = df_.loc[df_['AGE_GROUP_5'] == 'ALL']
    
    # Filter rows to keep only those with ORG_TYPE as 'ICB' or 'SUB_ICB_LOCATION_CODE'
    filtered_df = df[df['ORG_TYPE'].isin(['ICB', 'SUB_ICB_LOCATION_CODE'])].copy()

    #Dropping unused columns
    df_ = df_.drop(columns=['PUBLICATION','EXTRACT_DATE','ORG_TYPE','POSTCODE','SEX','ORG_CODE'])
    
    return df_

In [47]:
gp_list_df = catalog.get_catalog_entry_by_name(GP_PATIENTS_LIST_CATALOG)
patients_df = gp_list_df.load()
patients_df.head()
patients_df['ORG_TYPE'].unique()

array(['Comm Region', 'ICB', 'SUB_ICB_LOCATION_CODE', 'PCN', 'GP'],
      dtype=object)

In [24]:
patients_df.head()

,PUBLICATION,EXTRACT_DATE,ORG_TYPE,ORG_CODE,ONS_CODE,POSTCODE,SEX,AGE_GROUP_5,NUMBER_OF_PATIENTS
0,GP_PRAC_PAT_LIST,2024-09-01,Comm Region,Y56,E40000003,NaN,ALL,ALL,11051350
1,GP_PRAC_PAT_LIST,2024-09-01,Comm Region,Y56,E40000003,NaN,FEMALE,0_4,255861
2,GP_PRAC_PAT_LIST,2024-09-01,Comm Region,Y56,E40000003,NaN,FEMALE,10_14,300274
3,GP_PRAC_PAT_LIST,2024-09-01,Comm Region,Y56,E40000003,NaN,FEMALE,15_19,295834
4,GP_PRAC_PAT_LIST,2024-09-01,Comm Region,Y56,E40000003,NaN,FEMALE,20_24,384423


In [44]:
workforce_entry =  catalog.get_catalog_entry_by_name(GP_WORKFORCE_CATALOG)
workforce_df = workforce_entry.load()
workforce_df.head(3)
workforce_df['ICB_CODE'].unique()

array(['QJ2', 'QK1', 'QT1', 'QJM', 'QF7', 'QUE', 'QWO', 'QHL', 'QWU',
       'QUA', 'QYG', 'QE1', 'QOP', 'QMM', 'QJG', 'QHG', 'QM7', 'QMJ',
       'QRV', 'QMF', 'QT6', 'QWE', 'QOC', 'QNC', 'QUY', 'QH8', 'QNX',
       'QHM', 'QXU', 'QRL', 'QPM', 'QGH', 'QOQ', 'QJK', 'QR1', 'QSL',
       'QKK', 'QVV', 'QU9', 'QKS', 'QNQ', 'QOX', 'Unknown'], dtype=object)

In [22]:
workforce_df.head(3)

,YEAR,Month,COMM_REGION_CODE,COMM_REGION_NAME,ICB_CODE,ICB_NAME,SUB_ICB_CODE,SUB_ICB_NAME,DATA_SOURCE,UNIQUE_IDENTIFIER,STAFF_GROUP,DETAILED_STAFF_ROLE,STAFF_ROLE,COUNTRY_QUALIFICATION_AREA,COUNTRY_QUALIFICATION_GROUP,AGE_BAND,AGE_YEARS,GENDER,FTE
0,2024,8,Y60,Midlands,QJ2,NHS Derby and Derbyshire ICB,15M,NHS Derby and Derbyshire ICB - 15M,Provided,103041,GP,Salaried By Practice,Salaried GPs,UK,UK,30-34,30.0,Male,0.640000
1,2024,8,Y60,Midlands,QJ2,NHS Derby and Derbyshire ICB,15M,NHS Derby and Derbyshire ICB - 15M,Provided,172433,Direct Patient Care,Healthcare Assistant,Healthcare Assistants,Not Applicable,Not Applicable,55-59,55.0,Female,0.906667
2,2024,8,Y60,Midlands,QJ2,NHS Derby and Derbyshire ICB,15M,NHS Derby and Derbyshire ICB - 15M,Provided,179320,Admin/Non-Clinical,Receptionist,Receptionists,Not Applicable,Not Applicable,Under 25,23.0,Female,0.800000


In [ ]:
appointments_catalog_entry = catalog.get_catalog_entry_by_name(GP_APPOINTMENTS_CATALOG_NAME)
appointments_df = appointments_catalog_entry.load()
appointments_df.head()
appointments_df['SUB_ICB_LOCATION_ONS_CODE'].unique()

KeyError: 'ICB_CODE'